# Wafer sensor data: first look and cleaning

**Goal:** Handling messy data and building a model to find which signals correlate with failures

**Data:** Equipment sensor data and wafer test results from Infineon Technologies Austria and KAI, published on Zenodo (https://zenodo.org/records/4322353). Two pieces of equipment (56 sensors total), 176 readings per wafer, and a good/bad test result for each wafer.

**Notebook order:** this is notebook 1 of 5. Next: feature engineering, SPC, feature importance, modeling.

**Why I picked it:** The data contains sensor recordings from production equipment plus wafer test results. These recordings are helpful to find correlations between wafer test data and sensor data to identify the root cause. 

### 1. Load the files

In [1]:
import pandas as pd
from pathlib import Path

data_dir = Path("..")/"data"/"raw"
eq1 = pd.read_csv(data_dir/"equipment1.csv", sep=";")
eq2 = pd.read_csv(data_dir/"equipment2.csv", sep=";")
resp = pd.read_csv(data_dir/"response.csv", sep=";")

### 2. First look at every file

In [2]:
for name, df in [("eq1",eq1), ("eq2", eq2), ("resp", resp)]:
    print("====", name, df.shape)
    display(df.head(3))
    print(df.dtypes.value_counts())
    print("missing cells:", df.isna().sum().sum())
    print()

==== eq1 (170896, 27)


,lot,wafer,timestamp,sensor_1,sensor_2,sensor_3,sensor_4,sensor_5,sensor_6,sensor_7,...,sensor_15,sensor_16,sensor_17,sensor_18,sensor_19,sensor_20,sensor_21,sensor_22,sensor_23,sensor_24
0,lot2,1,timestamp_0,0.0,0.0,15.1604,105.4170,99.18,0.0,2.77728,...,0.0,0.0,7.0,200.0,151.891,0.0,47.5,0.0,-1.0,19.05
1,lot2,1,timestamp_1,0.0,0.0,15.0920,100.5100,98.18,0.0,2.77465,...,0.0,0.0,7.0,200.0,151.402,0.0,48.5,0.0,-1.0,0.99
2,lot2,1,timestamp_2,0.0,0.0,15.0432,95.3093,95.18,0.0,2.86352,...,0.0,0.0,7.0,200.0,149.815,0.0,48.0,0.0,-1.0,0.99


float64    24
str         2
int64       1
Name: count, dtype: int64
missing cells: 0

==== eq2 (232144, 35)


,lot,wafer,timestamp,sensor_25,sensor_26,sensor_27,sensor_28,sensor_29,sensor_30,sensor_31,...,sensor_47,sensor_48,sensor_49,sensor_50,sensor_51,sensor_52,sensor_53,sensor_54,sensor_55,sensor_56
0,lot2,1,timestamp_0,0.0,382.64,99.0,11.0,0.0,0.0,0.305503,...,2177.084,0.0,1.0,-0.1,1.8,199.7436,724.5978,0.000009,1.302417,1.00
1,lot2,1,timestamp_1,0.0,382.64,99.0,11.0,0.0,0.0,0.304918,...,2177.084,0.0,1.0,-0.1,1.8,199.6521,724.5978,0.000008,1.304668,1.18
2,lot2,1,timestamp_2,0.0,382.64,99.0,11.0,0.0,0.0,0.307562,...,2177.084,0.0,1.0,-0.1,1.8,199.7436,724.5978,0.000008,1.302417,0.93


float64    32
str         2
int64       1
Name: count, dtype: int64
missing cells: 0

==== resp (2638, 4)


,lot,wafer,response,class
0,lot2,1,1.0278,bad
1,lot2,2,0.3906,good
2,lot2,3,0.3906,good


str        2
int64      1
float64    1
Name: count, dtype: int64
missing cells: 0



The description says every wafer has 176 readings. Let's examine if it's correct.

### 3. Rows per wafer

In [3]:
for name, df in [("eq1", eq1),("eq2", eq2)]:
    per_wafer = df.groupby(["lot", "wafer"]).size()
    print(name, "| wafers:", len(per_wafer), "| lots:", df["lot"].nunique())
    print(per_wafer.describe()[["min", "50%", "max"]])
    print()

eq1 | wafers: 971 | lots: 40
min    176.0
50%    176.0
max    176.0
dtype: float64

eq2 | wafers: 1319 | lots: 54
min    176.0
50%    176.0
max    176.0
dtype: float64



Minimum, median and maximum are all 176 in both files. Equipment 1 has 971 wafers in 40 lots. Equipment 2 has 1,319 wafers in 54 lots. Notice that the number of wafers in equipment 1 is less than in equipment 2. I will carry on a feature step to examine the issue and decide what to do with this.

### 4. Is the response file one row per wafer?

In [4]:
print("rows:", len(resp))
print("unique (lot, wafer):", resp[["lot", "wafer"]].drop_duplicates().shape[0])
print("fully duplicated rows:", resp.duplicated().sum())
print(resp.groupby(["lot", "wafer"]).size().value_counts())

copies = resp.groupby(["lot", "wafer"])["response"].nunique()
print("wafers with more than one distinct response:", (copies > 1).sum())

rows: 2638
unique (lot, wafer): 1319
fully duplicated rows: 1319
2    1319
Name: count, dtype: int64
wafers with more than one distinct response: 0


There are 2,638 rows in response file but only 1,319 unique wafers. Every wafer appears exactly twice, and the two copies are identical. My decision is to drop the duplicated rows.

### 5. Clean the response file

In [5]:
resp_clean = resp.drop_duplicates().reset_index(drop=True)

# Stop with an error if any wafer still appears more than once
assert len(resp_clean) == resp_clean[["lot", "wafer"]].drop_duplicates().shape[0]

print(resp_clean.shape)
print(resp_clean["class"].value_counts())
print(resp_clean["class"].value_counts(normalize=True).round(3))

(1319, 4)
class
good    1055
bad      264
Name: count, dtype: int64
class
good    0.8
bad     0.2
Name: proportion, dtype: float64


### 6. Do the files cover the same wafers?

In [6]:
k1 = set(zip(eq1["lot"], eq1["wafer"]))
k2 = set(zip(eq2["lot"], eq2["wafer"]))
kr = set(zip(resp["lot"], resp["wafer"]))
print("eq1 wafers:", len(k1), "| eq2 wafers:", len(k2), "| resp wafers:", len(kr))
print("eq1 inside eq2?", k1 <= k2)
print("eq2 wafers with a response:", len(k2 & kr))


eq1 wafers: 971 | eq2 wafers: 1319 | resp wafers: 1319
eq1 inside eq2? True
eq2 wafers with a response: 1319


1,319 wafers in equipment 2 have 1,319 test results. The difference of (1319 - 971 = 348) wafers in equipment 1 needs to be resolved.

### 7. Class balance and lots

In [7]:
bad_rate = (resp_clean["class"] == "bad").groupby(resp_clean["lot"]).mean()
print(bad_rate.describe().round(3))
print("lots with zero bad wafer:", (bad_rate == 0).sum())
print(bad_rate.sort_values().tail(5).round(2))

count    54.000
mean      0.198
std       0.298
min       0.000
25%       0.000
50%       0.040
75%       0.230
max       1.000
Name: class, dtype: float64
lots with zero bad wafer: 25
lot
lot103    0.76
lot61     0.76
lot58     0.80
lot54     0.88
lot27     1.00
Name: class, dtype: float64


From Step 5, 264 of 1,319 wafers (20%) are bad. The number of bad wafers distributes unevenly over 54 lots of wafers. 25 lots have zero bad wafers, lot 27 has 100% bad wafers. I'm going to split data by lot in the modeling step.

### Summary:
- The data has no missing values. All sensors columns are numeric
- The response file has duplicates that double the number of rows
- There is a 348-wafer gap between equipment 1 and equipment 2 that will be resolved in the following steps
- Notice of uneven bad wafers across lots
  
**Next:** Notebook 2: Feature engineering